# CosyVoice3 클로닝 비교 (원본 vs 보호본) — Kaggle 버전

학습된 `best.pt` 로 임의 wav 한 개를 (a) 원본, (b) 보호본 으로 각각 CosyVoice3
zero-shot 클로닝한 뒤, 원래 화자와의 CAM++ 유사도를 비교한다.

- **원본으로 클론** → 유사도 **높음** (클로닝 성공)
- **보호본으로 클론** → 유사도 **크게 하락** 하면 방어 성공

## 실행 전 준비 (Kaggle 우측 패널)
1. **Settings → Accelerator → GPU T4 x2** (또는 P100)
2. **Settings → Internet → On**  ← git clone / gdown / 모델 다운로드에 필수

데이터는 **구글 드라이브에서 gdown으로** 가져온다(셀 2). Kaggle은 드라이브를
직접 마운트할 수 없으므로, `best.pt` 와 wav 1개를 드라이브 공유 링크로 받는다.
`campplus.onnx` 는 자동 다운로드되므로 따로 준비할 필요 없다.

> ⚠ CosyVoice 설치는 무겁다(5~10분). 한국어 텍스트 정규화 경고가 떠도 합성에는 보통 문제없다.

## 1. 코드 가져오기 + 기본 의존성
(인터넷이 켜져 있어야 한다.)

In [ ]:
import os, shutil
CODE_DIR = '/kaggle/working/voice_generator'
if os.path.isdir(CODE_DIR):
    shutil.rmtree(CODE_DIR)
!git clone https://github.com/Yena07/voice-generator-v3.git {CODE_DIR}
!pip install -q onnx onnx2torch soundfile

## 2. 구글 드라이브에서 파일 가져오기 (gdown)
Kaggle은 구글 드라이브를 직접 못 본다. 드라이브 **공유 링크**로 `best.pt` 와 wav 1개를 받는다.

**준비:** 드라이브에서 각 파일 우클릭 → 공유 → **"링크가 있는 모든 사용자"** 로 변경 → 링크 복사.
링크 `https://drive.google.com/file/d/`**`FILE_ID`**`/view` 에서 가운데 `FILE_ID` 만 아래에 붙여넣는다.
(`campplus.onnx` 는 HuggingFace에서 자동 다운로드되므로 준비 불필요.)

In [ ]:
!pip install -q gdown
import gdown, os, glob

# ── 드라이브 공유링크의 FILE_ID 입력 ──────────────────────────────
# 공유 설정: '링크가 있는 모든 사용자'. 링크 https://drive.google.com/file/d/<FILE_ID>/view
BEST_PT_ID = '여기에_best.pt_파일ID'      # ← 학습한 best.pt 의 FILE_ID
WAV_ID     = '여기에_wav파일_파일ID'        # ← 보호/클로닝 할 wav 1개의 FILE_ID
# ─────────────────────────────────────────────────────────────────

CHECKPOINT = '/kaggle/working/best.pt'
INPUT_WAV  = '/kaggle/working/sample.wav'
gdown.download(id=BEST_PT_ID, output=CHECKPOINT, quiet=False)
gdown.download(id=WAV_ID,     output=INPUT_WAV,  quiet=False)

# campplus.onnx 는 HuggingFace에서 자동 다운로드 (인터넷 On 필요)
CAMPPLUS = '/kaggle/working/campplus.onnx'
if not os.path.isfile(CAMPPLUS) or os.path.getsize(CAMPPLUS) < 1_000_000:
    !wget -q -O "{CAMPPLUS}" https://huggingface.co/model-scope/CosyVoice-300M/resolve/main/campplus.onnx

def _sz(p):
    return os.path.getsize(p) if os.path.isfile(p) else 'MISSING'
print('CODE_DIR  :', CODE_DIR)
print('CAMPPLUS  :', CAMPPLUS,   _sz(CAMPPLUS))
print('CHECKPOINT:', CHECKPOINT, _sz(CHECKPOINT))
print('INPUT_WAV :', INPUT_WAV,  _sz(INPUT_WAV))
assert os.path.isfile(CAMPPLUS),   'campplus 준비 실패 (인터넷 On 확인)'
assert os.path.isfile(CHECKPOINT), 'best.pt 다운로드 실패 → FILE_ID/공유설정(링크가 있는 모든 사용자) 확인'
assert os.path.isfile(INPUT_WAV),  'wav 다운로드 실패 → FILE_ID/공유설정 확인'

## 3. 보호본 생성 (infer.py)
원본에 보호 noise를 입혀 `/kaggle/working/protected.wav` 저장. 소리는 원본과 거의 같아야 정상.

In [ ]:
PROTECTED = '/kaggle/working/protected.wav'
!python "{CODE_DIR}/infer.py" \
    --checkpoint "{CHECKPOINT}" \
    --input "{INPUT_WAV}" \
    --output "{PROTECTED}"

from IPython.display import Audio, display
print('원본');   display(Audio(INPUT_WAV))
print('보호본'); display(Audio(PROTECTED))

## 4. CosyVoice repo clone + 의존성 설치 (5~10분)
`pip install -r requirements.txt` 를 한꺼번에 하면 ruamel.yaml/hyperpyyaml **버전 충돌로
전체 설치가 중단**된다. → **한 줄씩** 설치해 충돌 줄만 건너뛴다.

In [ ]:
import os, shutil, subprocess
os.chdir('/kaggle/working')
# 폴더가 있어도 비어있을 수 있으므로 requirements.txt 존재로 판단 (깡통 폴더면 지우고 재clone)
if not os.path.isfile('/kaggle/working/CosyVoice/requirements.txt'):
    if os.path.isdir('/kaggle/working/CosyVoice'):
        shutil.rmtree('/kaggle/working/CosyVoice')
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git
os.chdir('/kaggle/working/CosyVoice')
!git submodule update --init --recursive

# 핵심 과학 스택(numpy/scipy/protobuf/onnxruntime)은 CosyVoice가 낮추면 ABI 충돌 → SKIP 후 따로 처리.
# openai-whisper도 옛 핀은 빌드 실패 → SKIP 후 --no-deps로 설치.
SKIP = {'torch', 'torchaudio', 'numpy', 'numba', 'numba-cuda', 'scipy', 'pandas',
        'matplotlib', 'protobuf', 'onnxruntime', 'openai-whisper'}
fails = []
for raw in open('requirements.txt'):
    line = raw.strip()
    if not line or line.startswith(('#', '-')):
        continue
    name = line.split(';')[0].split('==')[0].split('>=')[0].split('<')[0].split('>')[0].strip().lower()
    if name in SKIP:
        continue
    if subprocess.run(['pip', 'install', '-q', line]).returncode != 0:
        fails.append(line)
print('설치 실패한 줄 (대개 무시 가능):', fails)

!pip install -q hyperpyyaml modelscope
!pip install -q onnx onnx2torch soundfile   # 비교용 패키지 재확인

# transformers / numpy2 호환 정리:
!pip uninstall -y torchao                                          # numpy 충돌 유발 → 제거
!pip install --force-reinstall --no-deps "protobuf==5.29.5"        # transformers는 protobuf 5.26+ 필요
!pip install -q -U onnxruntime                                     # 옛 onnxruntime은 numpy1용 → 최신으로
!pip install -q --no-deps openai-whisper                           # CosyVoice frontend가 import
!pip install -q tiktoken                                           # whisper 런타임 의존
!pip show protobuf | grep Version
print('⚠ [Run → Restart session] 후 섹션 4는 건너뛰고 섹션 5부터 실행')

## 5. CosyVoice3 모델 다운로드 (약 2GB)
`/kaggle/working` 에 저장. (인터넷 On 필요)

In [ ]:
from huggingface_hub import snapshot_download
COSY_DIR = '/kaggle/working/CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B'
snapshot_download('FunAudioLLM/Fun-CosyVoice3-0.5B-2512', local_dir=COSY_DIR)
print('downloaded to', COSY_DIR)

## 6. 프롬프트 대사 준비 + CosyVoice3 로드
zero-shot은 프롬프트 음성의 **대사**가 필요하다. 라벨 JSON(`발화정보.stt`)을
`/kaggle/input` 에서 자동으로 찾고, **못 찾으면 직접 입력**한다.

In [ ]:
import os, glob, json

# (1) 프롬프트 대사: 입력 wav와 같은 이름의 라벨 JSON에서 자동 추출 시도
PROMPT_TEXT = ''
stem = os.path.splitext(os.path.basename(INPUT_WAV))[0]
hits = glob.glob(f'/kaggle/input/**/{stem}.json', recursive=True)
if hits:
    with open(hits[0], encoding='utf-8') as f:
        PROMPT_TEXT = json.load(f).get('발화정보', {}).get('stt', '')

# (2) 자동 추출 실패 시: 아래 따옴표 안에 원본 음성에서 들리는 대사를 직접 입력
if not PROMPT_TEXT:
    PROMPT_TEXT = '요즘은 무선 청소기 안 쓰는 사람이 없더라'   # ← INPUT_WAV에서 들리는 대사로 교체
print('프롬프트 대사:', PROMPT_TEXT)

# (3) 합성할 목표 문장
TTS_TEXT = '안녕하세요. 이 음성은 음성 합성 모델로 만들어진 결과입니다.'

# (4) CosyVoice3 로드 (cwd와 무관하게 import되도록 repo 루트를 sys.path에 명시)
os.chdir('/kaggle/working/CosyVoice')
import sys, torchaudio
sys.path.insert(0, '/kaggle/working/CosyVoice')
sys.path.insert(0, '/kaggle/working/CosyVoice/third_party/Matcha-TTS')
assert os.path.isdir('/kaggle/working/CosyVoice/cosyvoice'), \
    'CosyVoice 폴더가 없음 → 섹션 4(설치)부터 다시 실행하세요.'
from cosyvoice.cli.cosyvoice import AutoModel
cosyvoice = AutoModel(model_dir=COSY_DIR)
print('CosyVoice3 로드 완료, sample_rate =', cosyvoice.sample_rate)

## 7. 원본 / 보호본으로 각각 클로닝

In [ ]:
import torchaudio

def clone(prompt_wav, out_path):
    sys_prompt = 'You are a helpful assistant.<|endofprompt|>' + PROMPT_TEXT
    for j in cosyvoice.inference_zero_shot(TTS_TEXT, sys_prompt, prompt_wav, stream=False):
        torchaudio.save(out_path, j['tts_speech'], cosyvoice.sample_rate)
        break   # 첫 세그먼트만 사용
    return out_path

CLONE_ORIG = '/kaggle/working/clone_from_original.wav'
CLONE_PROT = '/kaggle/working/clone_from_protected.wav'

print('원본으로 클로닝...');  clone(INPUT_WAV, CLONE_ORIG)
print('보호본으로 클로닝...'); clone(PROTECTED, CLONE_PROT)
print('완료')

## 8. 결과 들어보기
②는 원래 화자처럼, ③은 화자가 뭉개지거나 다른 목소리로 들리면 방어 성공.

In [ ]:
from IPython.display import Audio, display
print('① 원본(프롬프트 화자)');      display(Audio(INPUT_WAV))
print('② 원본으로 클론한 합성음');   display(Audio(CLONE_ORIG))
print('③ 보호본으로 클론한 합성음'); display(Audio(CLONE_PROT))

## 9. 화자 유사도 수치 비교 (CAM++)
원래 화자(원본 wav)와 각 합성음의 CAM++ 코사인 유사도를 계산한다.
합성음은 CosyVoice sample_rate이므로 16kHz로 리샘플 후 임베딩을 뽑는다.

In [ ]:
import sys, numpy as np, torch, soundfile as sf
from math import gcd
from scipy.signal import resample_poly

sys.path.insert(0, CODE_DIR)
from loss import campplus_embed_single, cosine_dist
import onnx, onnx2torch

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
cam = onnx2torch.convert(onnx.load(CAMPPLUS)).eval().to(dev)
for p in cam.parameters():
    p.requires_grad = False

def emb16k(path):
    a, sr = sf.read(path, dtype='float32', always_2d=True)
    a = a.mean(axis=1).astype(np.float32)
    if sr != 16000:
        g = gcd(16000, sr); a = resample_poly(a, 16000 // g, sr // g).astype(np.float32)
    a = np.clip(a, -1.0, 1.0)
    t = torch.from_numpy(a).unsqueeze(0).to(dev)
    with torch.no_grad():
        return campplus_embed_single(cam, t)

ref = emb16k(INPUT_WAV)
sim_orig = 1.0 - cosine_dist(emb16k(CLONE_ORIG), ref).item()
sim_prot = 1.0 - cosine_dist(emb16k(CLONE_PROT), ref).item()
red = (sim_orig - sim_prot) / max(sim_orig, 1e-6) * 100

print('=' * 46)
print(' 원래 화자와의 CAM++ 유사도 (1에 가까울수록 동일 화자)')
print('=' * 46)
print(f'  ② 원본으로 클론   : {sim_orig:.4f}   (높을수록 클로닝 성공)')
print(f'  ③ 보호본으로 클론 : {sim_prot:.4f}   (낮을수록 방어 성공)')
print(f'  유사도 감소율     : {red:.1f}%')
print('-' * 46)
print('  ✅ 방어 효과 있음' if sim_prot < sim_orig - 0.1 else '  ⚠ 방어 효과 미미 — 더 학습 필요')